In [13]:
from pyspark.sql import SparkSession
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler

import os
import logging
import datetime

spark = SparkSession.builder.getOrCreate()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
%%configure -f
{
    "conf": {
        "spark.driver.memory": "9G",
        "spark.executor.cores": "5",
        "spark.executor.instances": "30",
        "spark.executor.memory": "9G",
        "spark.sql.pivotMaxValues": "15000"
    }
}

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
2,application_1731491157434_0003,pyspark,idle,Link,Link,assumed-role_SSO_MKTG_STRAT_RTI_stage1540_mediaset_it,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1731491157434_0001,pyspark,idle,Link,Link,assumed-role_SSO_MKTG_STRAT_RTI_stage1540_mediaset_it,
2,application_1731491157434_0003,pyspark,idle,Link,Link,assumed-role_SSO_MKTG_STRAT_RTI_stage1540_mediaset_it,✔


In [5]:
PROJECT_NAME = "clustering_editoriale"

config = dict(
    version = 30,  # this should stay fixed, change only if you want to change the training period (which will also change labels, etc...)
    bucket = "mediaset-mktg",
    input_basedir = f"/gc/{PROJECT_NAME}/",
    output_basedir = f"/pf_1540/",
    env = "dev",
    min_tts_th = 300,  # drop unqualified fruitions that are below this threshold
    tts_monthly_th = 10800,  # drop months where a user's fruitions (over all brands) don't reach this threshold
    ## TODO: drop monthly_features config
    use_monthly_features = False,  # whether to average monthly brands into one, or keep one feature for each month (and brand)
    use_prop = False,  # use TTS proportion instead of absolute values
    ## drop features whose variance is below this threshold (0 or None to skip variance selection step)
#     variance_th = 0.01,  ## DON'T use variance_th, otherwise it's hard to ensure the same set of features in both training and testing set
    verbose = 1,
    window_size = 12,  # number of months to aggregate into a single feature
    training_yyyymm_cutoff = 202312,  # create training df up to this ym
    prediction_yyyymm = 202408,  # predict rolling window_size up to this yymm
)
output_basedir = config.get("output_basedir", f"/pf_1540/").strip("/")
input_basedir = config.get("input_basedir", f"gc/{PROJECT_NAME}").strip("/")
bucket = config.get("bucket", "mediaset-mktg")
env = config.get("env", "dev").lower()
version = int(config.get("version", 0))
min_tts_th = int(config.get("min_tts_th", 0))
tts_monthly_th = int(config.get("tts_monthly_th", 0))
use_monthly_features = config.get("use_monthly_features", False)
use_prop = config.get("use_prop", False)
# variance_th = float(config.get("variance_th", 0))
verbose = int(config.get("verbose", 1))
window_size = int(config.get("window_size", 12))
training_yyyymm_cutoff = int(config.get("training_yyyymm_cutoff"))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
if str(use_monthly_features).lower().startswith("f"):
    use_monthly_features = False
elif str(use_monthly_features).lower().startswith("t"):
    use_monthly_features = True
else:
    raise ValueError("Invalid value specified for option 'use_monthly_features'")

if str(use_prop).lower().startswith("f"):
    use_prop = False
elif str(use_prop).lower().startswith("t"):
    use_prop = True
else:
    raise ValueError("Invalid value specified for option 'use_prop'")

assert min_tts_th >= 0, "min_tts_th must be a positive number!"
assert version >= 0, "version must be a positive number!"

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
input_dir = f"s3://{bucket}/{input_basedir}/{env}/V{str(version).zfill(2)}"
output_dir = f"s3://{bucket}/{output_basedir}"
print(f"input_dir: {input_dir}")
print(f"output_dir: {output_dir}")

exp_name_verbose = f"monthly_tts_th={tts_monthly_th}__monthlyfeatures={use_monthly_features}__use_prop={use_prop}"  #__varsel={variance_th}"
print(f"exp_name_verbose: {exp_name_verbose}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

input_dir: s3://mediaset-mktg/gc/clustering_editoriale/dev/V30
output_dir: s3://mediaset-mktg/pf_1540
exp_name_verbose: monthly_tts_th=10800__monthlyfeatures=False__use_prop=False

In [8]:
fp = f"{input_dir}/dfs/2__sc_users_exploded_series_enriched"
df = spark.read.parquet(fp)
cnt = df.count()
print(f"{cnt:_}, partitions: {df.rdd.getNumPartitions()}")
df.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

433_444_099, partitions: 142
+----------------+--------------------+------+------+
|          series|            user_uid|   tts|    ym|
+----------------+--------------------+------+------+
|      terraamara|008a672e594642709...|  8670|202311|
|    uominiedonne|0014afa8ebca46018...|  3850|202311|
|pomeriggiocinque|00928d1c54ef482d9...|121984|202311|
| championsleague|0073b365ed3643179...| 20059|202311|
|    annaeicinque|006e28381bdf4bcb8...|    76|202311|
+----------------+--------------------+------+------+
only showing top 5 rows

In [9]:
yyyymms = sorted([e[0] for e in df.select("ym").distinct().collect()]) #P: Extract unique values of the 'ym' column from the DataFrame, sort them, and store them in the 'yyyymms' list.
yyyymms

#P: If 'training_yyyymm_cutoff' is not defined or is set to 0, assign it the maximum value of 'ym' from the DataFrame.
if training_yyyymm_cutoff is None or training_yyyymm_cutoff == 0:
    training_yyyymm_cutoff = df.select(F.max("ym")).collect()[0][0]
training_yyyymm_cutoff

#P: Find the index of 'training_yyyymm_cutoff' in the 'yyyymms' list and add 1 to it (to include the cutoff month)
idx = yyyymms.index(training_yyyymm_cutoff)+1

#P: Select a sequence of 'window_size' number of 'ym' values ending at 'training_yyyymm_cutoff'.
training_yyyymms = yyyymms[idx-window_size:idx]

#P: Ensure the length of 'training_yyyymms' is equal to 'window_size'. If not, raise an assertion error with a message.
assert len(training_yyyymms) == window_size, f"training_yyyymms length is {len(training_yyyymms)}, should be {window_size}"
training_yyyymms

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[202301, 202302, 202303, 202304, 202305, 202306, 202307, 202308, 202309, 202310, 202311, 202312]

In [11]:
"""
    Function to build and transform a DataFrame by applying various filters and aggregations.

    Args:
        df: Input DataFrame to be processed.
        min_tts_th: Minimum threshold for the 'tts' column. Rows with 'tts' below this value are filtered out (default is 300).
        use_monthly_features: Boolean flag indicating whether to include monthly features (default is False).
        use_prop: Boolean flag to indicate if 'tts' values should be reported as proportions (default is False).
        tts_monthly_th: Threshold for monthly 'tts' values to filter users by their activity level (default is 10800).

    Returns:
        A transformed DataFrame after filtering and aggregating the input 'df' based on the provided criteria.
"""

def build_df(df, min_tts_th: int = 300, use_monthly_features: bool = False, use_prop: bool = False, tts_monthly_th: int = 10800):

#              , vector_assembler_input_cols: List[str] = []):
    df = df.filter("user_uid != 'ghostery'")
    df = df.filter("series != 'unknown'")
    df = df.filter(F.col("tts") >= min_tts_th) #P: filter qualified fruitions

    def filter_user_by_monthly_tts(df, tts_monthly_th: int = 0):
    #P: Function to filter out users based on their monthly 'tts' values.

        #P: If the threshold is not set or <= 0, log a message and return the DataFrame as-is without filtering.
        if tts_monthly_th <= 0:
            log_and_print(f"TTS threshold is {tts_monthly_th}, won't filter df")
            return df

        ## filter out fruitions from months where users do not reach at least monthly_tts
        grp_cols = ["user_uid", "ym"] #P: grouping columns
        monthly_tts_by_user = df.groupBy(grp_cols).agg(
            F.sum("tts").alias("tts") #P: Aggregate 'tts' column to get total monthly 'tts' for each user.

        )

 #P: Retain only records from months where users have reached at least 'tts_monthly_th' threshold.
        df = df.join(
            monthly_tts_by_user.filter(f"tts >= {tts_monthly_th}").select(grp_cols), #P: Filter by threshold.
            grp_cols, #P: Join on 'user_uid' and 'ym' columns.
            "inner" #P: Use inner join to keep only matching records.
        )
#         prev_cnt = cnt
#         cnt = df.count()
#         prev_cnt_users = cnt_users
#         cnt_users = df.select(F.hll_sketch_estimate(F.hll_sketch_agg("user_uid")).alias("users")).collect()[0][0]
#         log_and_print(f"Dropping fruitions for months that don't reach {tts_monthly_th} monthly tts removed {prev_cnt - cnt:_} (-{round((abs(prev_cnt-cnt)) / prev_cnt * 100, 2)}%) records")
#         log_and_print(f"New users {cnt_users:_} (-{round((abs(prev_cnt_users-cnt_users)) / prev_cnt_users * 100, 2)}%)")

        return df

    def prepare_df(df):
        if use_prop:
            grp_cols = ["user_uid"]
            if use_monthly_features:
                grp_cols.append("ym")
            g = df.groupby(grp_cols).agg(
                F.sum(F.col("tts")).alias("tts_tot")
            )
            df = df.join(g, on=grp_cols, how="outer")
            df = df.withColumn("tts", F.col("tts") / F.col("tts_tot"))

        #P: our case (use_prop=F, use_monthly_features=F)
        else:
            if not use_monthly_features:
                ## Take the average TTS of active months
                df = df.groupby("user_uid", "series").agg(
                    (F.sum("tts") / F.countDistinct(F.col("ym"))).alias("tts")
                )
        pivot_col = F.concat(F.col("series"), F.lit("__"), F.col("ym")) if use_monthly_features else F.col("series") #P: in our case we take only series
        df = df.withColumn("pivot_col", pivot_col)
        df = df.groupby("user_uid").pivot("pivot_col").agg(
            (F.log(F.first("tts")) if not use_prop else F.first("tts")).cast("float")
        ) #P: in our case we transform in logaritms for better performance

        return df

    df = filter_user_by_monthly_tts(df, tts_monthly_th=tts_monthly_th)

    df = prepare_df(df)

    return df

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
#P:  (2023-01 / 2023-12)
train_df = build_df(
    df.filter(F.col("ym").isin(training_yyyymms)),
    min_tts_th=min_tts_th, use_monthly_features=use_monthly_features, use_prop=use_prop, tts_monthly_th=tts_monthly_th
)
#print(train_df.columns)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
#train_df.count() 6760334

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

6760334

In [18]:
# Calling sampling representative IDs
id = f"{output_dir}/sampling_idsandlabs"
sampling_idsandlabs = spark.read.parquet(id)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
# Sampling
sample1= train_df.join(sampling_idsandlabs, on="user_uid", how="inner")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
# Save the sample
sample1.coalesce(1).write.mode('overwrite').csv(f"{output_dir}/sample1.csv", header=True)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…